# Logistic Regression

By the end of this session, you will understand how logistic regression turns a linear combination of features into a probability for binary classification, and be able to fit and evaluate it within the leak-free workflow you already know.

# 1. Retrieval

1. In the linear-regression model $$ \hat y=wx+b $$ what do $w$ and $b$ control?

> Linear-regression finds the best fit for a straight line through data, and the equation of this straight line is made up of two parameters; $w$ controls the slope and $b$ controls the y-intercept

2. Suppose we extend that idea to many input features: $$ z=\mathbf{w}^\top\mathbf{x}+b $$ If $\mathbf{x}$ contains 30 features, what shape would you expect $\mathbf{w}$ to have, and what kind of quantity is $z$ for one observation?

> $\mathbf{w}$ whould have shape $(30,)$, so $\mathbf{w}\in\mathbb{R}^{30}$.
>
> $z$ represents a single linear score calculated from the observation $\mathbf{x}$. Each feature contributes according to its weight, and everything is combined into one scalar value 

3. Why should a `StandardScaler` be placed inside a pipeline before cross-validation rather than fitted to all of `X_train` first?

> Putting `StandardScaler()` inside the pipeline and passing this pipeline for cross-validation ensures that `StandardScaler` is fitted only on the four fitting folds. The fitted scaler is then used to transform both the fitting folds and the validation fold. This ensures that the validation fold is transformed using parameters learned elsewhere and the validation observations themselves do not help determine these parameters

4. Suppose a binary classifier gets: $$ TP=90,\quad FN=10 $$ What is its recall for the positive class, and what does that value mean in words?

> $\text{Recall}=\frac{TP}{TP+FN}=\frac{90}{90+10} = 0.9$
>
> This means that the proportion of correctly predicted positives to genuinely positive cases is 0.9

## 2. Why not just use linear regression for classification?

Suppose our target is binary: $$y\in \set{0,1}$$

A first idea might be to fit: $$\hat{y} = \mathbf{w}^\top\mathbf{x}+b$$

and interpret $\hat{y}$ as the probability that the observation belongs to class 1.

The problem is that a linear function is unbounded: $$-\infty<\mathbf{w}^\top\mathbf{x}+b<\infty$$

So it could predict: $$\hat{y} = -0.7$$

sor: $$\hat{y}=1.8$$

Neither makese sense as a probability.

We want to keep the useful linear combination: $$z=\mathbf{w}^\top\mathbf{x}+b$$

but transform $z$ into something restricted to: $$0<p<1$$

That is what the **sigmoid function** does.

## 3. The sigmoid

The sigmoid is: $$\sigma(z)=\frac{1}{1 + e^{-z}}$$

Its input can be any real number: $$-\infty<z<\infty$$

but its output is always between 0 and 1: $$0<\sigma(z)<1$$

Some useful values: $$\sigma(0)=0.5$$

For large positive $z$: $$\sigma(z)\rarr 1$$

For large negative $z$: $$\sigma(z)\rarr 0$$

So logistic regression takes the linear score: $$z = \mathbf{w}^\top\mathbf{x}+b$$

and passes it through the sigmoid: $$\boxed{P(y=1\mid\mathbf{x})=\sigma(\mathbf{w}^\top\mathbf{x}+b)}$$

Thats the central logisitic-regression model.

## 4. From probability to predicted class

A probability score isn't yet a class prediction.

With the conventional threshold of: $$0.5$$

we predict: $$ \hat y= \begin{cases} 1 & \text{if }P(y=1\mid\mathbf{x})\geq0.5\\ 0 & \text{otherwise} \end{cases} $$

Because $\sigma(0)=0.5$, this is equivalent to:

$$ z\geq0\Rightarrow\hat y=1 $$ $$ z<0\Rightarrow\hat y=0 $$

We'll not tune that threshold today. For this session, $0.5$ is simply the default decision rule.

## 5. Conceptual check

1. Suppose two observations give linear scores: $$ z_A=3 $$ $$ z_B=-1 $$ Without calculating exact probabilities, which observation receives the higher $P(y=1\mid\mathbf{x})$, and why?

> $z_A$ would recieve the higher $P(y=1\mid\mathbf{x})$ because we know $\sigma(0)=0.5$, therefore $0.5 < \sigma(z_A) < 1$ while $0 < \sigma(z_B) < 0.5$.
>
> Since the sigmois is an increasing function: $$z_A>z_B\quad\Rightarrow\quad \sigma(z_A)>\sigma(z_B)$$

2. Why can't we interpret the raw value $$ z=\mathbf{w}^\top\mathbf{x}+b $$ itself as a probability?

> Because the output of this linear combination can take any value between $\pm\infty$, while a probability must lie between $(0,1)$

3. If logistic regression gives: $$ P(y=1\mid\mathbf{x})=0.73 $$ what class would the default $0.5$ threshold predict?

> This observation $\mathbf{x}$ would be classified as belonging to class $1$

## 6. What do the logistic-regression weights mean? 

For multiple features, our moidel begins with: 

$$z = w_1x_1 + w_2x_2 +\dots+w_dx_d+b$$

and then: $$p=P(y=1\mid\mathbf{x})=\sigma(z)$$

Because the sigmoid is increasing, anything that increases **$z$ also increases the predicted probability of class 1**.

So, holding all the other features constant:
- $w_j>0$: increasing $x_j$ increases the model's probability of class 1;
- $w_j<0$: increasing $x_j$ decreases the model's probability of class 1;
- larger $\mid w_j\mid$: that feature has a stronger effect on the model's linear score, all else equal.

For example: $$z = 2x_1-0.5x_2 +b$$

Increasing $x_1$ pushes the prediction towards class 1.

Increasing $x_2$ pushes it towards class 0.

But there's an important qualififcation:

> A coefficient is **not** a fixed change in probability.

A coefficient of $2$ does not mean increasing this feature by one increases the porbability by 2 or by 20%.

Thats because the coefficient acts on $z$, and then $z$ passes through the nonlinear sigmoid.

## 7. Why is it called *logistic* regressions? Log-odds

Theres another way to interpret that linear score.

Suppose: $$p=P(y=1\mid\mathbf{x})$$

The odds of class 1 are: $$\text{odds}=\frac{p}{1-p}$$

For example, if: $$p=0.8$$

then: $$\text{odds}=\frac{0.8}{0.2}=4$$

which we can read as odds of **4 to 1** in favour of class 1.

Now take the logarithm: $$\log{\left(\frac{p}{1-p}\right)}$$

This is called the **log-odds**, or **logit**.

For logistic regression, something near happens: $$\boxed{\log{\left(\frac{p}{1-p}\right)}=\mathbf{w}^\top\mathbf{x}+b}$$

So the raw score $z$ that we've been discussing is actually the model's **log-odds of class 1**.

That gives the coefficient a precise interpretation:

> Increasing $x_j$ by one unit changes the log-odds of class 1 by $w_j$, holding the other features constant.

For today's purposes, the most important interpretation is still simpler:

$$w_j>0 \Rightarrow\text{feature pushes towards class 1}$$
$$w_j<0 \Rightarrow\text{feature pushes towards class 0}$$

We'll use that when we inspect one fitted coefficient later.

## 8. How does logistic regression learn the weights?

We now have a model that produces probabilities, but we still need some way to decide which values of: $$\mathbf{b}, b$$ are good.

With linear regression, we used the **mean squared error**.

For logigistic regression, the standard loss is **binary cross-entropy**, also called the **log loss**. 

For one observation: $$L=-[y\log{(p)}+(1-y)\log{(1-p)}]$$

where:
- $y$ is the true label, either $0$ or $1$;
- $p$ is the model's predicted probability of class 1.

### Example

#### If the true label is 1: 

$$y=1$$

Then: $$L=-\log{(p)}$$

So if the model correctly gives class 1 as a high probability: $$p=0.9$$

then: $$L=-\log(0.9)\approx0.105$$

Small loss.

But if it confidently gets it wrong: $$p=0.1$$

then: $$L=-\log{(0.1)}\approx2.303$$

Much larger loss

#### If the true label is 0:

Now: $$y=0$$

and the first term disappears: $$L=-\log{(1-p)}$$

If the model predicts: $$p=0.1$$

then it is correctly assigning a high probability to *class 0*: $$1-p = 0.9$$

so the loss is small.

But if: $$p=0.9$$

the model is confidently predicting class 1 when the truth is class 0, so the loss is large.

### The central idea

Log loss rewards probabilities that put high probability onm the correct class and **strongly penalises confident wrong predictions**.

During training, logistic regression chooses $\mathbf{w}$ and $b$ to minimise the loss across the training observations.

This connects directly back to the optimisation work you already covered:

$$\mathbf{w}, b\rarr\text{predicted probabilities}\rarr\text{loss}\rarr\text{optimisation}$$

## 9. Conceptual check

1. A logistic-regression feature has coefficient: $$ w_j=-1.4 $$ Holding everything else constant, what does increasing that feature do to:
    - the linear score $z$;
    - the predicted probability of class 1?

    > Increasing $x_j$ decreases the linear score $z$ and therefore decreases the models probability of class 1

2. For a true class-1 observation, which prediction should receive the larger log loss? $$ p=0.95 $$ or $$ p=0.20 $$ Explain why without calculating it.

> $p = 0.20$ would recieve a larger log loss, since the $(1-y)$ in the log-loss function becomes $0$, therefore $$L=-y\log(p)$$ and $-\log(0.20) > -\log(0.95)$

3. A model predicts: $$ P(y=1\mid\mathbf{x})=0.8 $$ What are the odds of class 1?

> $\text{odds}=\frac{0.8}{1-0.8} = 4$
>
> The odds of class 1 are 4:1

4. Why would it be incorrect to say that a logistic-regression coefficient of $+0.5$ means “a one-unit feature increase raises the probability of class 1 by 0.5”?

> Because the coefficient acts on $z$, and then $z$ passes through the nonlinear sigmoid to calculate probability

## 10. Main implementation task

Continue using the Breast Cancer Wisconsin with exactly the same split:
```python
test_size = 0.2
random_state = 42
stratify = y
```
### Part A — Build the model
Create a pipeline containing:
- `StandardScaler`
- `LogisticRegression`

Use the default logistic-regression model settings for now. Do not tune anything.

### Part B — Cross-validation
Using only `X_train` and `y_train`, perform 5-fold cross-validation and report:
- the five accuracy scores;
- mean CV accuracy;
- standard deviation of CV accuracy

Then compare the result with the 5-NN result from Session 3: $$\text{kNN CV mean}\approx 0.967$$

### Part C — Final held-out evaluation
Fit the logistic-regression pipeline to the whole training set, then predict `X_test`. 

Report:
- test accuracy;
- recall/sensitivity for malignant, remembering that malignant is label 0.

Then brefly answer:

> Based on CV versus the previous scaled k-NN model, which model would you have selected before seeing either final test result?

### Part D — One coefficient interpretation
From the fitted pipeline:
1. inspect the logistic-regression coefficient array;
2. report its shape;
3. choose one feature;
4. report its feature name and coefficient
5. interpret only the sign of that coefficient.

In [36]:
import numpy as np
from sklearn import metrics
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict, cross_val_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

breast_cancer = load_breast_cancer()

X = breast_cancer.data
y = breast_cancer.target

X_train, X_test, y_train, y_test = train_test_split(
  X,
  y, 
  test_size=0.2,
  random_state=42,
  stratify=y
)

# LogisticRegression classifier with StandardScaler pipeline
log_reg_scaled = Pipeline([
  ('scaler', StandardScaler()),
  ('classifier', LogisticRegression())
])

## 5-fold cross-validation
log_reg_scaled_CV_scores = cross_val_score(log_reg_scaled, X_train, y_train, cv=5)

print(f'accuracy scores: {log_reg_scaled_CV_scores}')
print(f'mean: {np.mean(log_reg_scaled_CV_scores): .3f}')
print(f'std: {np.std(log_reg_scaled_CV_scores): .3f}')

print('\nLogistic-regression mean CV accuracy is higher than 5-nearest-neighbours mean CV accuracy from session 3 (approx 0.967)')

# Fit logistic-regression pipeline to wholle trainng set
log_reg_scaled.fit(X_train, y_train)

# Predict X_test
y_pred = log_reg_scaled.predict(X_test)

n_correct_predictions = np.count_nonzero(y_pred == y_test)
accuracy = n_correct_predictions / len(y_pred)
recall = metrics.recall_score(y_test, y_pred, pos_label=0)

print(f'\nFinal test accuracy: {accuracy: .3f}')
print(f'Recall for malignant classification: {recall:.3f}')


accuracy scores: [0.96703297 0.97802198 0.96703297 1.         0.98901099]
mean:  0.980
std:  0.013

Logistic-regression mean CV accuracy is higher than 5-nearest-neighbours mean CV accuracy from session 3 (approx 0.967)

Final test accuracy:  0.982
Recall for malignant classification: 0.976


### Part C answer:

> Based on mean CV scores from X_train data, I would have chosen the logistic-regression modelling since it scored a mean CV of 0.980 compared to approximatley 0.967 for 5-NN modelling

In [37]:
log_reg_model = log_reg_scaled.named_steps['classifier']
coefficients = log_reg_model.coef_

print('Coefficients shape:', coefficients.shape)
print(coefficients)

feature_index = 5
feature_name = breast_cancer.feature_names[feature_index]
print(f'\nInspecting feature {feature_index}: {feature_name}')
print(f'w_{feature_index} = {coefficients[0, feature_index]}')
print(f'Holding the other features constant, an increase in standardised {feature_name} pushes the model towards benign classification')

Coefficients shape: (1, 30)
[[-0.51147901 -0.55269775 -0.47629789 -0.54105924 -0.21247927  0.64834159
  -0.60210291 -0.70415649 -0.16723273  0.19973173 -1.08296534  0.24882301
  -0.54433323 -0.92910402 -0.16027571  0.64722656  0.16056252 -0.44378424
   0.36049156  0.43789426 -0.94761615 -1.25508804 -0.76322007 -0.9477559
  -0.74662481  0.05551412 -0.82315065 -0.95368636 -0.93918141 -0.1872508 ]]

Inspecting feature 5: mean compactness
w_5 = 0.6483415872660625
Holding the other features constant, an increase in standardised mean compactness pushes the model towards benign classification
